In [0]:
# Databricks notebook source

from __future__ import annotations

import uuid

# ============================================================
# Widgets
# ============================================================
dbutils.widgets.text("catalog", "phc_txdot")
catalog = dbutils.widgets.get("catalog").strip()

# ============================================================
# Constants
# ============================================================
SCRIPT_NAME = "461_build_gold_vw_fact_project_scope_mix.py"
RUN_ID = str(uuid.uuid4())

SOURCE_TABLE = f"{catalog}.gold.fact_project_scope_mix"
TARGET_VIEW = f"{catalog}.gold.vw_fact_project_scope_mix"
RUN_LOG_TABLE = f"{catalog}.staging.pipeline_run_log"

# ============================================================
# Helpers
# ============================================================
def sql_escape(value: str) -> str:
    return value.replace("'", "''")


def log_run(status: str, row_count: int | None, message: str) -> None:
    row_count_sql = "NULL" if row_count is None else str(row_count)
    spark.sql(f"""
        INSERT INTO {RUN_LOG_TABLE}
        VALUES (
              '{RUN_ID}'
            , '{SCRIPT_NAME}'
            , '{sql_escape(status)}'
            , {row_count_sql}
            , '{sql_escape(message)}'
            , current_timestamp()
        )
    """)

# ============================================================
# Build gold.vw_fact_project_scope_mix
# ============================================================
try:
    table_exists = spark.sql(f"""
        SELECT COUNT(*) AS cnt
        FROM {catalog}.information_schema.tables
        WHERE table_schema = 'gold'
          AND table_name = 'fact_project_scope_mix'
    """).collect()[0]["cnt"]

    if table_exists == 0:
        raise RuntimeError(
            "Required table not found: gold.fact_project_scope_mix\n"
            "Run the project scope mix gold fact build first."
        )

    spark.sql(f"DROP VIEW IF EXISTS {TARGET_VIEW}")

    spark.sql(f"""
    CREATE VIEW {TARGET_VIEW} AS
    SELECT
          project_scope_mix_key                          AS `Project Scope Mix Key`
        , project_key                                    AS `Project Key`
        , md5(COALESCE(item_work_category, ''))          AS `Item Work Category Key`
        , project_id                                     AS `Project ID`
        , item_work_category                             AS `Item Work Category`
        , estimate_item_row_count                        AS `Estimate Item Row Count`
        , estimate_distinct_spec_count                   AS `Estimate Distinct Spec Count`
        , estimate_category_amount                       AS `Estimate Category Amount`
        , estimate_item_rollup_total                     AS `Estimate Item Rollup Total`
        , max_project_engineer_total                     AS `Project Engineer Total`
        , engineer_project_total_source                  AS `Engineer Project Total Source`
        , estimate_category_pct_of_project_total         AS `Estimate Category Pct Of Project Total`
        , estimate_category_pct_of_item_rollup_total     AS `Estimate Category Pct Of Item Rollup Total`
        , winning_bid_item_row_count                     AS `Winning Bid Item Row Count`
        , winning_bid_distinct_spec_count                AS `Winning Bid Distinct Spec Count`
        , winning_bid_category_amount                    AS `Winning Bid Category Amount`
        , winning_bid_item_rollup_total                  AS `Winning Bid Item Rollup Total`
        , lowest_bid_amount_in_project                   AS `Lowest Bid Amount In Project`
        , winning_bid_category_pct_of_low_bid_total      AS `Winning Bid Category Pct Of Low Bid Total`
        , winning_bid_category_pct_of_item_rollup_total  AS `Winning Bid Category Pct Of Item Rollup Total`
        , winning_bid_vs_estimate_category_amount_diff   AS `Winning Bid Vs Estimate Category Amount Diff`
        , winning_bid_vs_estimate_category_amount_ratio  AS `Winning Bid Vs Estimate Category Amount Ratio`
        , category_present_in_both_estimate_and_winning_bid AS `Category Present In Both Estimate And Winning Bid`
        , category_estimate_only                         AS `Category Estimate Only`
        , category_winning_bid_only                      AS `Category Winning Bid Only`
    FROM {SOURCE_TABLE}
    """)

    row_count = spark.sql(f"""
        SELECT COUNT(*) AS row_count
        FROM {TARGET_VIEW}
    """).collect()[0]["row_count"]

    log_run(
        "SUCCESS",
        row_count,
        "Built gold.vw_fact_project_scope_mix successfully."
    )

    print("=" * 70)
    print("Built gold.vw_fact_project_scope_mix")
    print(f"Row count: {row_count:,}")
    print("=" * 70)

    summary = spark.sql(f"""
        SELECT
              COUNT(*) AS total_rows
            , COUNT(CASE WHEN `Project Key` IS NOT NULL THEN 1 END) AS project_key_rows
            , COUNT(CASE WHEN `Item Work Category Key` IS NOT NULL THEN 1 END) AS item_work_category_key_rows
        FROM {TARGET_VIEW}
    """).collect()[0]

    print(
        {
            "total_rows": summary["total_rows"],
            "project_key_rows": summary["project_key_rows"],
            "item_work_category_key_rows": summary["item_work_category_key_rows"],
        }
    )

except Exception as exc:
    log_run("FAILED", None, str(exc))
    raise